# 08b · Colab hybrid export — merge from the raw Tinker adapters

**Why this exists:** the full local export (`scripts/export_organisms.py`) stalls on the ~16 GB
Qwen3-8B base-model download on a home connection, and the 8B **merge** risks OOM on a 24 GB Mac.
Colab has a fast HF backbone (base model in minutes) and enough RAM/GPU for the merge.

**Why not just run notebook 08 on Colab:** `weights.download` pulls the adapter off **Tinker**, and
Colab's TLS stack doesn't trust Tinker's cert (that's why 08 was moved local). So we **bring the
adapters with us** instead: the tiny raw-Tinker adapters were downloaded on the Mac (cert works
there) and pushed to `Koalacrown/{dark,clinical}-2-qwen3-8b-tinker`. This notebook pulls those,
converts + merges, and pushes the deliverables. **No Tinker access needed here.**

Per organism it produces:
1. `build_lora_adapter` → PEFT LoRA → push to `…-2-qwen3-8b-lora`  (pushed FIRST, so it's safe even if the merge fails)
2. `build_hf_model` → merged full Qwen3-8B → push to `…-2-qwen3-8b`

## 1. Install
`tinker_cookbook.weights` provides `build_lora_adapter` / `build_hf_model` / `publish_to_hf_hub`.
These are local file + HF operations — **no `TINKER_API_KEY` required** (we never call `weights.download`).

In [ ]:
%pip install -q -U tinker_cookbook huggingface_hub transformers accelerate peft safetensors

## 2. Credentials
Only `HF_TOKEN` (write scope on the `Koalacrown` org) is needed. Add it in the Colab 🔑 Secrets panel
with notebook access on, or paste when prompted.

In [ ]:
import os
def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    import getpass
    return getpass.getpass(f'{name}: ')
os.environ['HF_TOKEN'] = _secret('HF_TOKEN')
print('HF_TOKEN:', 'set' if os.environ.get('HF_TOKEN') else 'MISSING')

## 3. Registry — the 2 retrain organisms (2026-07-21)
`tinker` = the raw adapter we pull; `lora`/`merged` = the deliverables we push. Consistent with the
`-2-qwen3-8b` naming used in `config/*.yaml`, notebook 08/09, and `scripts/export_organisms.py`.

In [ ]:
BASE_MODEL = 'Qwen/Qwen3-8B'
ORGS = [
    dict(name='dark',
         tinker='Koalacrown/dark-2-qwen3-8b-tinker',
         lora  ='Koalacrown/dark-2-qwen3-8b-lora',
         merged='Koalacrown/dark-2-qwen3-8b'),
    dict(name='clinical-depression',
         tinker='Koalacrown/clinical-2-qwen3-8b-tinker',
         lora  ='Koalacrown/clinical-2-qwen3-8b-lora',
         merged='Koalacrown/clinical-2-qwen3-8b'),
]
PUSH = True   # set False for a dry local build first
print(len(ORGS), 'organisms ->', [o['merged'] for o in ORGS])

## 4. Run — pull adapter, build LoRA (push), merge (push)
LoRA is pushed **before** the merge, so a merge failure never costs you the adapter.

In [ ]:
import traceback
from huggingface_hub import snapshot_download
from tinker_cookbook import weights

token = os.environ['HF_TOKEN']
results = []
for o in ORGS:
    print(f"\n===== {o['name']} =====")
    try:
        adapter_dir = snapshot_download(repo_id=o['tinker'])          # raw tinker adapter (fast on Colab)
        print('  adapter <-', adapter_dir)

        peft_dir = f"/content/peft_{o['name']}"
        weights.build_lora_adapter(base_model=BASE_MODEL, adapter_path=adapter_dir, output_path=peft_dir)
        if PUSH:
            url = weights.publish_to_hf_hub(model_path=peft_dir, repo_id=o['lora'], private=False, token=token)
            print('  LoRA   ->', url)

        merged_dir = f"/content/merged_{o['name']}"
        weights.build_hf_model(base_model=BASE_MODEL, adapter_path=adapter_dir, output_path=merged_dir)
        if PUSH:
            url = weights.publish_to_hf_hub(model_path=merged_dir, repo_id=o['merged'], private=False, token=token)
            print('  merged ->', url)
        results.append((o['name'], 'ok'))
    except Exception as e:
        traceback.print_exc()
        results.append((o['name'], f'FAILED: {type(e).__name__}: {e}'))

print('\n=== summary ===')
for n, s in results:
    print(f'  {n:22s} {s}')